# 🚀 Two-Agent News Sentiment Analyzer with Agent-to-Agent Delegation

This notebook implements an agent-to-agent delegation workflow for financial sentiment analysis using AutoGen's **Nested Chats**:
1. Python fetches raw articles and queries the FAISS vector database for calibration examples.
2. The user initiates a chat with the **Senior Sentiment Analyst (CIO) Agent**.
3. The CIO Agent automatically triggers a nested chat, delegating the scoring task to the **Sentiment Scorer Agent**.
4. The Scorer Agent analyzes the articles, calculates scores, and returns them to the CIO Agent.
5. The CIO Agent aggregates the scores, averages the sentiment, and returns the final JSON report back to the user.

## 🛡️ Multi-Layered Safety Guardrails (Loop & Hallucination Prevention)

This workflow is protected by three layers of safety to prevent conversational auto-reply loops and LLM hallucinations under empty prompt inputs:

1. **Orchestration Layer**:
   * **Location**: `User_Proxy` configuration.
   * **Mechanism**: Sets `max_consecutive_auto_reply=0` to ensure that after receiving the CIO's report, the `User_Proxy` immediately halts the conversation instead of sending an empty string to keep it going. It also uses robust termination signature checks (`aggregate_score`, `sentiment_score`).

2. **Backend Handler Layer**:
   * **Location**: `sentiment/functions/tools/custom_reply.py`.
   * **Mechanism**: Implements an input check inside `custom_nested_chat_reply`. If the incoming message is empty or empty-like (`[]`), it returns immediately with a termination message, completely bypassing LLM execution to block invalid runs.

3. **Prompt Guideline Layer**:
   * **Location**: `sentiment/prompts/sentiment_prompt.txt`.
   * **Mechanism**: Enforces a strict LLM guideline under rules instructing the model to return a clean empty list `[]` and `TERMINATE` if it is given a blank or empty articles input, preventing it from hallucinating mock examples.

In [1]:
import sys
import os
from dotenv import load_dotenv

# Ensure the current directory is in the python path for importing modules
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

sentiment_dir = os.path.dirname(notebook_dir)
if sentiment_dir not in sys.path:
    sys.path.insert(0, sentiment_dir)
'''
project_root = os.path.dirname(sentiment_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
'''
# Load environment variables from .env.local
load_dotenv("../.env.local")

True

In [2]:
import json
import datetime
import pandas as pd
import autogen
from finrobot.agents.workflow import FinRobot
from autogen import UserProxyAgent
from functions.aggregator.aggregator import fetch_aggregate_all_news
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Import refactored utility functions
from functions.utils.read_and_clean import read_file_content, extract_and_clean_response
from functions.utils.build import build_vector_store
from functions.tools.prepare_articles, assign_label import prepare_articles, assign_label
from functions.agents import create_scorer_agent, create_cio_agent, create_decomposition_agent
from functions.tools.openbb import fetch_etf_holdings_from_openbb
from functions.utils.formulas import calculate_raw_sentiment, calculate_portfolio_sentiment, normalize_weights
from functions.tools.custom_reply import custom_nested_chat_reply, extract_json_array

# Read and clean environment variables
nvidia_embedding_model = os.getenv("NVIDIA_EMBEDDING_MODEL", "nvidia/nv-embed-v1").strip('"\' ')
nvidia_base_model = os.getenv("NVIDIA_BASE_MODEL", "").strip('"\' ')
nvidia_api_endpoint = os.getenv("NVIDIA_API_ENDPOINT", "https://integrate.api.nvidia.com/v1").strip('"\' ')
nvidia_api_key = os.getenv("NVIDIA_API_KEY", "").strip('"\' ')

print(f"Initializing NVIDIA Embeddings wrapper ({nvidia_embedding_model})...")
embeddings = NVIDIAEmbeddings(
    model=nvidia_embedding_model,
    nvidia_api_key=nvidia_api_key,
    base_url=nvidia_api_endpoint
)

d:\PartnaStudio\sentinel\stack\FinRobot-IntentChain\sentiment\venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")
d:\PartnaStudio\sentinel\stack\FinRobot-IntentChain\sentiment\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing NVIDIA Embeddings wrapper (nvidia/nv-embed-v1)...


C:\Users\19178\AppData\Local\Temp\ipykernel_9056\3075562704.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
from functions import llm_config, base_llm_config


HF Model Name: curiousily/Llama-3-8B-Instruct-Finance-RAG:featherless-ai
HF Base URL: https://router.huggingface.co/v1
HF API Key exists: True


In [4]:
ticker = "SPY"
news_limit = 3  # Score top 3 articles per entity
holdings = 5    # Top 5 constituents if ETF

In [5]:
# Build vector store
db = build_vector_store("../data/financial_sentiment.csv", embeddings, limit_rows=300)

# Instantiate the UserProxyAgent
user_proxy = UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "") and "TERMINATE" in x.get("content", ""),
    max_consecutive_auto_reply=15,
    code_execution_config={"use_docker": False}
)

Loading dataset: ../data/financial_sentiment.csv...
Indexing 300 records into FAISS vector database...
[+] FAISS Local Vector Store created successfully!


In [6]:
from autogen import register_function

# Instantiate scorer, CIO and Decomposition agents
scorer_agent = create_scorer_agent(
    prompt_path="../prompts/sentiment_prompt.txt",
    schema_path="../schema_json/scorer_schema.json",
    llm_config=llm_config
)

cio_agent = create_cio_agent(
    prompt_path="../prompts/cio_prompt.txt",
    schema_path="../schema_json/sentiment_schema.json",
    output_schema_path="../schema_json/cio_output_schema.json",
    scored_articles_path="../schema_json/cio_scored_articles.json",
    llm_config=base_llm_config
)

decomp_agent = create_decomposition_agent(
    prompt_path="../prompts/decomposition_prompt.txt",
    schema_path="../schema_json/decomposition_schema.json",
    example_path="../schema_json/decomposition_example.json",
    llm_config=llm_config
)

# Register tools
register_function(
    fetch_etf_holdings_from_openbb,
    caller=decomp_agent,
    executor=user_proxy,
    name="fetch_etf_holdings_from_openbb",
    description="Fetch underlying constituents and weights for an ETF tracker."
)

register_function(
    calculate_raw_sentiment,
    caller=cio_agent,
    executor=user_proxy,
    name="calculate_raw_sentiment",
    description="Calculates the confidence-weighted average sentiment score of scored articles for a single asset."
)

register_function(
    calculate_portfolio_sentiment,
    caller=cio_agent,
    executor=user_proxy,
    name="calculate_portfolio_sentiment",
    description="Aggregates effective ticker sentiments by portfolio holding weight."
)

register_function(
    normalize_weights,
    caller=cio_agent,
    executor=user_proxy,
    name="normalize_weights",
    description="Normalizes a dictionary of weights so that their sum equals 1.0."
)

register_function(
    assign_label,
    caller=cio_agent,
    executor=user_proxy,
    name="assign_label",
    description="Assigns a sentiment classification label based on the calculated sentiment score."
)


In [ ]:
def notebook_news_sentiment_custom_reply(chat_queue, recipient, messages, sender, config):
    """Custom reply handler that orchestrates ETF decomposition and constituent batch scoring."""
    # 1. First-Turn Loop Guard (prevent re-running news pipeline on tool execution replies)
    if len(messages) > 1:
        return False, None

    print("\n[+] Step 1: Running ETF Decomposition check...")
    decomp_chat = recipient._get_chats_to_run([chat_queue[0]], recipient, messages, sender, config)
    res_decomp = autogen.initiate_chats(decomp_chat)
    decomp_summary = res_decomp[0].summary
    
    decomp_data = extract_json_array(decomp_summary)
    
    all_articles = []
    is_etf = False
    constituents = []
    
    if isinstance(decomp_data, dict) and not decomp_data.get("error_flag", True) and decomp_data.get("constituents"):
        candidate_constituents = decomp_data["constituents"]
        if len(candidate_constituents) == 1 and candidate_constituents[0].get("ticker", "").upper() == ticker.upper():
            print(f"[*] Decomposition returned target ticker {ticker} only. Treating as single stock fallback.")
            is_etf = False
        elif len(candidate_constituents) > 0:
            constituents = candidate_constituents[:holdings]
            is_etf = True

    if is_etf and constituents:
        print(f"[+] Decomposed ETF. Top {len(constituents)} constituents to analyze: {[c['ticker'] for c in constituents]}")
        
        for const in constituents:
            c_ticker = const["ticker"]
            print(f"[*] Fetching news for constituent: {c_ticker}...")
            try:
                df_c_news = fetch_aggregate_all_news(symbol=c_ticker, limit=100)
                if df_c_news.empty:
                    print(f"[-] No articles found for constituent {c_ticker}. Skipping.")
                    continue
                prepared = prepare_articles(df_c_news, db, limit=news_limit)
                for art in prepared:
                    art["ticker"] = c_ticker
                all_articles.extend(prepared)
            except Exception as ex:
                print(f"[-] Error processing constituent {c_ticker}: {ex}. Skipping.")
    else:
        print(f"[*] Ticker {ticker} is a single stock or failed decomposition. Processing fallback...")
        try:
            df_news = fetch_aggregate_all_news(symbol=ticker, limit=100)
            if not df_news.empty:
                prepared = prepare_articles(df_news, db, limit=news_limit)
                for art in prepared:
                    art["ticker"] = ticker
                all_articles.extend(prepared)
        except Exception as ex:
            print(f"[-] Error processing ticker {ticker}: {ex}.")

    # Send the decomposition report/status to the nesting agent's conversation history
    if is_etf and constituents:
        decomp_report = (
            f"ETF Decomposition Report:\n"
            f"{json.dumps(decomp_data, indent=2)}"
        )
    else:
        decomp_report = (
            f"ETF Decomposition Report: This asset is a single stock. No constituents found."
        )
        
    recipient.send(
        message=decomp_report,
        recipient=sender,
        request_reply=False,
        silent=True
    )

    if not all_articles:
        recipient.send(
            message="[]",
            recipient=sender,
            request_reply=False,
            silent=True
        )
        return False, None

    # Batch score the articles in cycles of 5
    batch_size = 5
    batch_results = []
    
    for i in range(0, len(all_articles), batch_size):
        batch = all_articles[i:i + batch_size]
        print(f"[*] Batching scoring pipeline: processing articles {i+1} to {min(i+batch_size, len(all_articles))} of {len(all_articles)}")
        
        temp_chat_config = chat_queue[1].copy()
        temp_chat_config["message"] = (
            "Please score the following articles according to your instructions:\n\n"
            f"{json.dumps(batch, indent=2)}\n\n"
            "Respond with the list of scored articles."
        )
        
        scorer_chat = recipient._get_chats_to_run([temp_chat_config], recipient, messages, sender, config)
        res_scorer = autogen.initiate_chats(scorer_chat)
        
        summary_content = res_scorer[-1].summary
        scored_data = extract_json_array(summary_content)
        if scored_data is not None:
            batch_results.append(scored_data)
        else:
            try:
                clean_content = summary_content
                if clean_content.endswith("TERMINATE"):
                    clean_content = clean_content[:-9].strip()
                batch_results.append(json.loads(clean_content))
            except Exception:
                pass

    # Merge results and send back to CIO history
    from functions.tools.custom_reply import merge_scored_results
    merged_result = merge_scored_results(batch_results)
    scorer_summary = json.dumps(merged_result, indent=2)
    
    recipient.send(
        message=scorer_summary,
        recipient=sender,
        request_reply=False,
        silent=True
    )
    
    return False, None


Fetching consolidated news feed for AAPL...
[+] Fetching OpenBB YFINANCE...
[+] Fetching OpenBB TMX...
    -> OpenBB TMX Error: Results not found.
[+] Fetching OpenBB FMP...
    -> OpenBB FMP Error: 
[Error] -> Unauthorized FMP request -> 402 -> Restricted Endpoint: This endpoint is not available under your current subscription please visit our subscription page to upgrade your plan at https://financialmodelingprep.com/
[+] Fetching OpenBB TIINGO...
    -> OpenBB TIINGO Error: 
[Error] -> Unauthorized Tiingo request -> You do not have permission to access the News API
[+] Fetching OpenBB BIZTOC...
[+] Fetching OpenBB BENZINGA...
[+] Fetching Custom ALPHA-VANTAGE...
[+] Fetching Custom NEWS_API...
[+] Fetching Custom SEEKING-ALPHA...
Seeking Alpha Error: 403 Client Error: Forbidden for url: https://seeking-alpha-api.p.rapidapi.com/news/v2/list-by-symbol?symbol=AAPL&size=100
[+] Fetching Custom NASDAQ...
[+] Fetching Custom FINVIZ...

[+] Triggering Agent-to-Agent Delegation Chat...
User

In [8]:
print(final_report_msg)

{
  "ticker": "AAPL",
  "metadata": {
    "timestamp": "2026-06-16T23:14:00.000Z",
    "article_count": 2
  },
  "articles": [
    {
      "title": "Apple's OLED MacBook push raises stakes for BOE and Samsung display race",
      "source": "Finviz Source",
      "published_at": "2026-06-16 23:14:00",
      "sentiment_label": "Positive",
      "sentiment_score": 0.83,
      "confidence": 0.92,
      "risk_factors": ["regulatory_headwind", "supply_chain_disruption"],
      "reasoning_summary": "Apple's OLED MacBook push raises stakes for BOE and Samsung display race",
      "flagged": false,
      "flag_reason": null
    },
    {
      "title": "Apple moves private cloud compute to third party with Google Cloud AI",
      "source": "Finviz Source",
      "published_at": "2026-06-16 22:55:00",
      "sentiment_label": "Positive",
      "sentiment_score": 0.92,
      "confidence": 0.95,
      "risk_factors": ["regulatory_headwind", "supply_chain_disruption"],
      "reasoning_summary": "Ap